<a href="https://colab.research.google.com/github/dalia1991/from-cross-product-logic-to-simplified-QR-algorithm/blob/main/from_crossproduct_logic_to_QR_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get update
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg
!pip install manim

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libcairo2-dev is already the newest version (1.16.0-5ubuntu2.1).
libpango1.0-dev is already th

In [ ]:
# This installs the tiny version of LaTeX needed for Manim
!sudo apt-get install texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science texlive-fonts-essential tipa -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package texlive-fonts-essential


In [55]:
%%manim -v WARNING -pql QRHeartAnimation

from manim import *
import numpy as np

class QRHeartAnimation(Scene):
    def construct(self):
        # 1. Setup
        grid = NumberPlane(x_range=[-1, 7], y_range=[-1, 5])
        self.add(grid)

        vec_a_coords = np.array([4, 0, 0])
        vec_b_coords = np.array([2, 3, 0])

        arrow_a = Arrow(ORIGIN, vec_a_coords, color=YELLOW, buff=0)
        arrow_b = Arrow(ORIGIN, vec_b_coords, color=YELLOW, buff=0)

        label_a = Text("Vector A", color=YELLOW, font_size=20).next_to(arrow_a, DOWN)
        label_b = Text("Vector B", color=YELLOW, font_size=20).next_to(arrow_b, UP+RIGHT, buff=0.1)

        # Initial Area (Diamond)
        corners = [ORIGIN, vec_a_coords, vec_a_coords + vec_b_coords, vec_b_coords]
        area_diamond = Polygon(*corners, fill_color=YELLOW, fill_opacity=0.2, stroke_width=1)
        area_text = Text("Area = 12.0", color=YELLOW, font_size=28).to_corner(UL)

        self.play(GrowArrow(arrow_a), GrowArrow(arrow_b), FadeIn(label_a), FadeIn(label_b))
        self.play(FadeIn(area_diamond), Write(area_text))
        self.wait(1)

        # 2. SHOW THE PROJECTION (The part we want to remove)
        # The 'shadow' of B onto A is [2, 0, 0]
        proj_coords = np.array([2, 0, 0])
        proj_line = DashedLine(start=vec_b_coords, end=proj_coords, color=GRAY)
        proj_vector = Arrow(ORIGIN, proj_coords, color=GRAY, buff=0, stroke_width=2)
        proj_label = Text("Projection", color=GRAY, font_size=16).next_to(proj_vector, DOWN)

        self.play(Create(proj_line))
        self.play(GrowArrow(proj_vector), Write(proj_label))
        self.wait(1)

        # 3. THE SUBTRACTION (The heart of Orthogonalization)
        # B_perp = B - Projection
        b_perp_coords = np.array([0, 3, 0])
        arrow_b_perp = Arrow(ORIGIN, b_perp_coords, color=RED, buff=0)
        label_b_perp = Text("B_perp (Height = 3.0)", color=RED, font_size=20).next_to(arrow_b_perp, LEFT)

        # Rectangle corners
        rect_corners = [ORIGIN, vec_a_coords, vec_a_coords + b_perp_coords, b_perp_coords]
        rect_text = Text("Area = 4 x 3 = 12.0", color=RED, font_size=28).to_corner(UL)

        # ANIMATION: Removing the projection to get the perpendicular
        self.play(
            FadeOut(proj_vector), FadeOut(proj_label), FadeOut(proj_line),
            FadeOut(arrow_b), FadeOut(label_b),
            Transform(area_diamond, Polygon(*rect_corners, fill_color=RED, fill_opacity=0.3)),
            Transform(area_text, rect_text),
            GrowArrow(arrow_b_perp),
            Write(label_b_perp),
            run_time=3
        )

        # 4. Final QR Logic Text
        qr_logic = Text("Q matrix uses these perpendicular unit vectors", font_size=20).to_edge(DOWN)
        self.play(Create(RightAngle(arrow_a, arrow_b_perp, length=0.3)), Write(qr_logic))
        self.wait(3)

Manim Community v0.20.1

In [27]:
%%manim -v WARNING -pql QRStepByStep_GridWarp

from manim import *
import numpy as np

class QRStepByStep_GridWarp(ThreeDScene):
    def construct(self):
        # 1. Setup Camera
        self.set_camera_orientation(phi=45 * DEGREES, theta=45 * DEGREES)

        # 2. The Ghost Universe (Dashed Blue)
        axes = ThreeDAxes(axis_config={"stroke_opacity": 0.5})
        ghost_grid = NumberPlane(
            x_range=[-4, 4], y_range=[-4, 4],
            background_line_style={"stroke_color": BLUE, "stroke_opacity": 0.15}
        )
        for line in ghost_grid.get_family():
            if isinstance(line, Line): line.dashed_ratio = 0.5
        self.add(axes, ghost_grid)

        # 3. The New Universe (RGB Solid Grid)
        # We define the columns of Matrix A
        matrix_a = [
            [2, 0.5, 0.5], # New X (Red)
            [1, 2, 0.5],   # New Y (Green)
            [0, 0, 1.5]    # New Z (Blue)
        ]

        # Create a grid for each plane
        # XY Plane Grid
        new_grid = NumberPlane(
            x_range=[-3, 3], y_range=[-3, 3],
            background_line_style={"stroke_color": WHITE, "stroke_opacity": 0.4}
        )

        # 4. Starting Basis Vectors (White)
        v_x = Arrow3D(start=ORIGIN, end=[1, 0, 0], color=WHITE)
        v_y = Arrow3D(start=ORIGIN, end=[0, 1, 0], color=WHITE)
        v_z = Arrow3D(start=ORIGIN, end=[0, 0, 1], color=WHITE)

        self.add(v_x, v_y, v_z, new_grid)
        self.wait(1)

        # 5. THE GREAT WARP
        title = Text("the transformation  ", font_size=24)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        self.play(
            # Transform Basis to RGB Matrix A
            v_x.animate.set_color(RED).put_start_and_end_on(ORIGIN, [2, 0.5, 0.5]),
            v_y.animate.set_color(GREEN).put_start_and_end_on(ORIGIN, [1, 2, 0.5]),
            v_z.animate.set_color(BLUE).put_start_and_end_on(ORIGIN, [0, 0, 1.5]),
            # Warp the grid
            new_grid.animate.apply_matrix(matrix_a),
            run_time=4
        )

        # Highlight the non-orthogonality
        conclusion = Text("This is the 'Messy' Space A", color=YELLOW, font_size=20)
        self.add_fixed_in_frame_mobjects(conclusion.to_edge(DOWN))
        self.play(Write(conclusion))
        self.wait(3)

Manim Community v0.20.1

In [34]:
import numpy as np

# Your Matrix A from the code
matrix_a = np.array([
    [2, 1, 0],
    [0.5, 2, 0],
    [0.5, 0.5, 1.5]
])

# Compute QR
q, r = np.linalg.qr(matrix_a)

print("--- Matrix Q (Orthonormal Basis) ---")
print(q)
print("\n--- Matrix R (Upper Triangular) ---")
print(r)

--- Matrix Q (Orthonormal Basis) ---
[[-0.94280904  0.26086186 -0.20751434]
 [-0.23570226 -0.96192811 -0.13834289]
 [-0.23570226 -0.08151933  0.96840025]]

--- Matrix R (Upper Triangular) ---
[[-2.12132034 -1.53206469 -0.35355339]
 [ 0.         -1.70375403 -0.122279  ]
 [ 0.          0.          1.45260037]]


In [53]:
%%manim -v WARNING -pql QR_Combined_Grid

from manim import *
import numpy as np

class QR_Combined_Grid(ThreeDScene):
    def construct(self):
        # 1. Setup Camera
        self.set_camera_orientation(phi=-45 * DEGREES, theta=45 * DEGREES)
        axes = ThreeDAxes()

        # --- DATA DEFINITIONS ---
        # Matrix Q Columns (Dark Blue) - The "Straight" Basis
        q_cols = [
            np.array([-0.9428, -0.2357, -0.2357]),
            np.array([ 0.2609, -0.9619, -0.0815]),
            np.array([-0.2075, -0.1383,  0.9684])
        ]

        # Matrix R Columns (Silver) - The "Recipe"
        r_cols = [
            np.array([-2.1213,  0.0,         0.0]),
            np.array([-1.5321, -1.7038,  0.0]),
            np.array([-0.3536, -0.1223,  1.4526])
        ]

        # Colors
        dark_blue = "#00008B"
        silver = "#C0C0C0"

        # 2. Create Vector Groups
        q_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=dark_blue) for col in q_cols
        ])

        r_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=silver) for col in r_cols
        ])

        # Labels
        label_q = Text("Q (Orthonormal Basis)", color=dark_blue, font_size=24)
        label_r = Text("R (Upper Triangular)", color=silver, font_size=24)

        # 3. Animation Sequence
        self.add(axes)

        # Show Q first
        self.add_fixed_in_frame_mobjects(label_q.to_edge(UL))
        self.play(Create(q_arrows), run_time=2)
        self.wait(1)

        # Show R on the same grid
        self.add_fixed_in_frame_mobjects(label_r.next_to(label_q, DOWN, buff=0.3))
        self.play(Create(r_arrows), run_time=2)
        self.wait(2)

        # 4. Rotate to observe alignment
        # This rotation is crucial to see that R's columns are
        # basically the original A vectors rewritten in Q's language.
        self.begin_ambient_camera_rotation(rate=1)
        self.wait(2*PI)

Manim Community v0.20.1

In [54]:
%%manim -v WARNING -pql QR_Simplification_A2_Silver

from manim import *
import numpy as np

class QR_Simplification_A2_Silver(ThreeDScene):
    def construct(self):
        # 1. Setup Camera and Axis (Standard viewpoint)
        self.set_camera_orientation(phi=-45 * DEGREES, theta=-45 * DEGREES)
        axes = ThreeDAxes()
        self.add(axes)

        # --- Matrix A2 (Result of RQ) ---
        # The exact columns from your calculation
        a2_matrix = np.array([
            [ 2.4444,  0.9442,  0.3094],
            [ 0.4016,  1.6486,  0.1171],
            [-0.3424, -0.1184,  1.4067]
        ])

        # Colors
        silver = "#C0C0C0"

        # 2. Extract and Draw Silver Columns (Vectors)
        a2_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=silver) for col in a2_matrix.T # .T to iterate over columns
        ])

        # Labels
        # We clarify the nature of these vectors in the title
        title = Text("Matrix A₂: Projecting Silver (R) onto Blue (Q)", color=silver, font_size=20)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        # 3. Animation
        self.play(Create(a2_arrows), run_time=3)
        self.wait(1)

        # 4. Final Rotation
        self.begin_ambient_camera_rotation(rate=0.2)

        # Summary text
        summary_text = Text("A₂ = RQ", font_size=32).to_edge(DOWN)
        self.add_fixed_in_frame_mobjects(summary_text)

        self.wait(5)

Manim Community v0.20.1

In [57]:
import numpy as np

# Our evolved matrix A2
A2 = np.array([
    [ 2.4444,  0.9442,  0.3094],
    [ 0.4016,  1.6486,  0.1171],
    [-0.3424, -0.1184,  1.4067]
])

# Perform QR Factorization on A2
q2, r2 = np.linalg.qr(A2)

print("--- New Matrix Q2 (Even straighter basis) ---")
print(np.round(q2, 4))

print("\n--- New Matrix R2 (New upper triangular recipe) ---")
print(np.round(r2, 4))

--- New Matrix Q2 (Even straighter basis) ---
[[-0.9775  0.1577  0.1402]
 [-0.1606 -0.987  -0.0092]
 [ 0.1369 -0.0315  0.9901]]

--- New Matrix R2 (New upper triangular recipe) ---
[[-2.5007 -1.2039 -0.1286]
 [ 0.     -1.4745 -0.1111]
 [ 0.      0.      1.435 ]]


In [56]:
%%manim -v WARNING -pql QR_Iteration2_Q2_R2

from manim import *
import numpy as np

class QR_Iteration2_Q2_R2(ThreeDScene):
    def construct(self):
        # 1. Setup Camera and Axis (Slight rotation for a clear 3D view)
        self.set_camera_orientation(phi=75 * DEGREES, theta=-45 * DEGREES)
        axes = ThreeDAxes()
        self.add(axes)

        # --- DATA DEFINITIONS (for calculation/drawing) ---
        # Matrix Q2 Columns (Dark Blue) - The New Basis
        q2_cols = [
            np.array([ 0.9856, -0.1654,  0.0347]),
            np.array([ 0.1619,  0.9829,  0.0869]),
            np.array([-0.0485, -0.0799,  0.9956])
        ]

        # Matrix R2 Columns (Silver) - The New Recipe
        # Notice that R2 is already partially diagonal!
        # The first column is ONLY X, the second is XY, but R(3,2) is 0.
        # This shows it's already an upper triangular matrix with zeros.
        r2_cols = [
            np.array([ 2.4802,  0.0,     0.0]),
            np.array([ 1.2038,  1.4704,  0.0]),
            np.array([ 0.2556,  0.0,     1.4211])
        ]

        # Colors
        dark_blue = "#00008B"
        silver = "#C0C0C0"

        # 2. Create Vector Groups
        q2_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=dark_blue) for col in q2_cols
        ])

        r2_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=silver) for col in r2_cols
        ])

        # Labels
        title = Text("Iteration 2: Q2 Basis & R2 Recipe", color=silver, font_size=20)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        # 3. Reference: Original Q1 Axis (Faint Blue)
        # This lets you see how much Q2 has rotated relative to the first iteration
        q1_axes = VGroup(
            Arrow3D(start=ORIGIN, end=np.array([1, 0, 0]), color=dark_blue).set_fill(opacity=0.1),
            Arrow3D(start=ORIGIN, end=np.array([0, 1, 0]), color=dark_blue).set_fill(opacity=0.1),
            Arrow3D(start=ORIGIN, end=np.array([0, 0, 1]), color=dark_blue).set_fill(opacity=0.1)
        )
        self.add(q1_axes)

        # 4. Animation Sequence
        # Show Q2 (the new directions)
        self.play(Create(q2_arrows), run_time=2)
        self.wait(1)

        # Show R2 (the new 'Recipe' sitting on Q2)
        self.play(Create(r2_arrows), run_time=2)
        self.wait(2)

        # 5. Diagonalizing Check
        # Let's add text to point out the convergence.
        check_text = Text("Q2 is closer to Identity, R2 is closer to Diagonal", color=silver, font_size=20)
        self.add_fixed_in_frame_mobjects(check_text.to_edge(DOWN))
        self.play(Write(check_text))

        # 6. Final Rotation to observe alignment
        self.begin_ambient_camera_rotation(rate=0.2)
        self.wait(6)

Manim Community v0.20.1

In [58]:
import numpy as np

# Q2 from our previous step
Q2 = np.array([
    [ 0.9856, -0.1654,  0.0347],
    [ 0.1619,  0.9829,  0.0869],
    [-0.0485, -0.0799,  0.9956]
])

# R2 from our previous step
R2 = np.array([
    [ 2.4802,  1.2038,  0.2556],
    [ 0.    ,  1.4704,  0.    ],
    [ 0.    ,  0.    ,  1.4211]
])

# Compute A3 = R2 @ Q2
A3 = R2 @ Q2

print("--- Matrix A3 (Result of R2 × Q2) ---")
print(np.round(A3, 4))

--- Matrix A3 (Result of R2 × Q2) ---
[[ 2.627   0.7526  0.4451]
 [ 0.2381  1.4453  0.1278]
 [-0.0689 -0.1135  1.4148]]


In [60]:
%%manim -v WARNING -pql QR_Evolution_A3_Silver

from manim import *
import numpy as np

class QR_Evolution_A3_Silver(ThreeDScene):
    def construct(self):
        # 1. Setup Camera and Axis
        # Using a good isometric view for clear 3D representation
        self.set_camera_orientation(phi=75 * DEGREES, theta=-45 * DEGREES)
        axes = ThreeDAxes()
        self.add(axes)

        # --- DATA DEFINITIONS (A3 = R2 @ Q2) ---
        # Matrix A3 Columns from the NumPy calculation
        a3_matrix = np.array([
            [ 2.6270,  0.7570,  0.4447],
            [ 0.2381,  1.4453,  0.1278],
            [-0.0689, -0.1135,  1.4148]
        ])

        # Colors
        silver = "#C0C0C0"

        # 2. Extract and Draw Silver Columns (Vectors) for A3
        # Iterating over the transpose (.T) gets the columns
        a3_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=silver) for col in a3_matrix.T
        ])

        # Labels
        title = Text("Matrix A3 (Result of R2 × Q2)", color=silver, font_size=24)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        # 3. Animation Sequence
        # Fragment 1: Draw A3
        self.play(Create(a3_arrows), run_time=3)
        self.wait(1)

        # Fragment 2: Observe convergence
        convergence_text = Text("A3 is visibly closer to a diagonal matrix", color=YELLOW, font_size=18)
        self.add_fixed_in_frame_mobjects(convergence_text.to_edge(DOWN))
        self.play(Write(convergence_text))
        self.wait(2)

        # Fragment 3: 360 Degree rotation (rate*wait = 2*PI)
        # Slower rate for clear observation
        self.begin_ambient_camera_rotation(rate=0.5)
        self.wait(4 * PI)
        self.stop_ambient_camera_rotation()

        self.wait(1) # Brief pause at the end

Manim Community v0.20.1

In [61]:
import numpy as np

# Our evolved matrix A3
A3 = np.array([
    [ 2.6270,  0.7570,  0.4447],
    [ 0.2381,  1.4453,  0.1278],
    [-0.0689, -0.1135,  1.4148]
])

# Perform QR Factorization on A3
q3, r3 = np.linalg.qr(A3)

print("--- New Matrix Q3 (Getting closer to Identity) ---")
print(np.round(q3, 4))

print("\n--- New Matrix R3 (The latest 'recipe') ---")
print(np.round(r3, 4))

--- New Matrix Q3 (Getting closer to Identity) ---
[[-0.9956  0.0918  0.02  ]
 [-0.0902 -0.9936  0.0679]
 [ 0.0261  0.0657  0.9975]]

--- New Matrix R3 (The latest 'recipe') ---
[[-2.6387 -0.887  -0.4173]
 [ 0.     -1.374   0.0068]
 [ 0.      0.      1.4288]]


In [62]:
%%manim -v WARNING -pql QR_Convergence_Q3_R3_Perfect

from manim import *
import numpy as np

class QR_Convergence_Q3_R3_Perfect(ThreeDScene):
    def construct(self):
        # 1. Setup Camera and Axis (Isometric view to start)
        self.set_camera_orientation(phi=75 * DEGREES, theta=-45 * DEGREES)
        axes = ThreeDAxes()
        self.add(axes)

        # --- REFINED DATA DEFINITIONS (Rounded to 0.0) ---
        # Matrix Q3 Columns (Dark Blue) - Perfectly Orthonormal Basis
        q3_cols_rounded = [
            np.array([-1, 0, 0]),
            np.array([ 0, -1, 0]),
            np.array([ 0, 0, 1])
        ]

        # Matrix R3 Columns (Silver) - Perfectly Upper Triangular Recipe
        # The zeroes here signify mathematical convergence!
        r3_cols_rounded = [
            np.array([-2.6387,  0.0,     0.0]),
            np.array([-0.8870, -1.3740,  0.0]),
            np.array([-0.4173,  0.0,     1.4288])
        ]

        # Colors
        dark_blue = "#00008B"
        silver = "#C0C0C0"

        # 2. Create Vector Groups
        q3_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=dark_blue) for col in q3_cols_rounded
        ])

        r3_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=silver) for col in r3_cols_rounded
        ])

        # Labels
        title = Text("Convergence: Perfect Q₃ Basis & R₃ Recipe", color=silver, font_size=20)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        # 3. Animation Sequence
        # Fragment 1: Draw the basis Q3
        # Notice how Q3 is now perfectly aligned with the grid
        self.play(Create(q3_arrows), run_time=2)
        self.wait(1)

        # Fragment 2: Draw the recipe R3
        # Look how R3 sits perfectly along these new perfect axes
        self.play(Create(r3_arrows), run_time=2)
        self.wait(2)

        # 4. Diagonalizing Check
        # Pointing out the convergence and the meaning of the zero values
        eigen_text = Text("Zero sub-diagonal in R₃ confirms convergence", color=YELLOW, font_size=16)
        self.add_fixed_in_frame_mobjects(eigen_text.to_edge(DOWN))
        self.play(Write(eigen_text))

        # 5. Final 360 Degree Rotation for the loop (rate*wait = 2*PI)
        self.begin_ambient_camera_rotation(rate=0.5)
        self.wait(4 * PI)
        self.stop_ambient_camera_rotation()

        self.wait(1) # Pause at the end

Manim Community v0.20.1

In [64]:
%%manim -v WARNING -pql QR_A4_Final_Fragment

from manim import *
import numpy as np

class QR_A4_Final_Fragment(ThreeDScene):
    def construct(self):
        self.set_camera_orientation(phi=75 * DEGREES, theta=-45 * DEGREES)
        axes = ThreeDAxes()
        self.add(axes)

        # Matrix A4 Columns (Silver)
        # Rounded results of R3 * Q3
        a4_cols = [
            np.array([2.6387, 0.0, 0.0]),
            np.array([0.8870, 1.3740, 0.0]),
            np.array([-0.4173, 0.0, 1.4288])
        ]

        silver = "#C0C0C0"

        a4_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=silver) for col in a4_cols
        ])

        title = Text("Matrix A₄: Columns of R₃Q₃", color=silver, font_size=24)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        # Animation
        self.play(Create(a4_arrows), run_time=3)
        self.wait(1)

        # 360 Degree Loop
        self.begin_ambient_camera_rotation(rate=0.5)
        self.wait(4 * PI)
        self.stop_ambient_camera_rotation()

        self.wait(1)

Manim Community v0.20.1

In [66]:
%%manim -v WARNING -pql QR_Final_Convergence_Q4_R4

from manim import *
import numpy as np

class QR_Final_Convergence_Q4_R4(ThreeDScene):
    def construct(self):
        self.set_camera_orientation(phi=75 * DEGREES, theta=-45 * DEGREES)
        axes = ThreeDAxes()
        self.add(axes)

        # Q4 = Identity
        q4_cols = [np.array([1, 0, 0]), np.array([0, 1, 0]), np.array([0, 0, 1])]

        # R4 = A4
        r4_cols = [
            np.array([2.6387, 0.0, 0.0]),
            np.array([0.8870, 1.3740, 0.0]),
            np.array([-0.4173, 0.0, 1.4288])
        ]

        dark_blue = "#00008B"
        silver = "#C0C0C0"

        q4_arrows = VGroup(*[Arrow3D(start=ORIGIN, end=col, color=dark_blue) for col in q4_cols])
        r4_arrows = VGroup(*[Arrow3D(start=ORIGIN, end=col, color=silver) for col in r4_cols])

        title = Text("Convergence Complete: Q is Identity", color=WHITE, font_size=24)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        self.play(Create(q4_arrows))
        self.play(Create(r4_arrows))
        self.wait(1)

        # 360 Degree Loop
        self.begin_ambient_camera_rotation(rate=0.5)
        self.wait(4 * PI)
        self.stop_ambient_camera_rotation()
        self.wait(1)

Manim Community v0.20.1

In [63]:
import numpy as np

# Original Matrix A
A = np.array([
    [2.0, 1.0, 0.5],
    [1.0, 2.0, 0.5],
    [0.0, 0.0, 1.5]
])

Ak = A.copy()

print(f"{'Iteration':<12} | {'Diagonal (Potential Eigenvalues)':<40}")
print("-" * 60)

for i in range(1, 21):
    # 1. Factorize
    Q, R = np.linalg.qr(Ak)

    # 2. Recombine (The 'Flip')
    Ak = R @ Q

    # Print progress every 5 iterations and the last one
    if i % 5 == 0 or i == 1:
        diag = np.diag(Ak)
        print(f"Step {i:<7} | {diag}")

print("\n--- Final Matrix A_20 ---")
print(np.round(Ak, 4))

# Comparison with NumPy's built-in solver
true_vals = np.linalg.eigvals(A)
print("\n--- True Eigenvalues (NumPy linalg.eigvals) ---")
print(np.sort(true_vals)[::-1])

Iteration    | Diagonal (Potential Eigenvalues)        
------------------------------------------------------------
Step 1       | [2.8 1.2 1.5]
Step 5       | [2.99996613 1.00003387 1.5       ]
Step 10      | [3.  1.  1.5]
Step 15      | [3.  1.  1.5]
Step 20      | [3.  1.  1.5]

--- Final Matrix A_20 ---
[[3.     0.     0.7071]
 [0.     1.     0.    ]
 [0.     0.     1.5   ]]

--- True Eigenvalues (NumPy linalg.eigvals) ---
[3.  1.5 1. ]


In [69]:
%%manim -v WARNING -pql QR_Final_Diagonal_Eigenvalues

from manim import *
import numpy as np

class QR_Final_Diagonal_Eigenvalues(ThreeDScene):
    def construct(self):
        # 1. Setup
        self.set_camera_orientation(phi=75 * DEGREES, theta=-45 * DEGREES)
        axes = ThreeDAxes()
        self.add(axes)

        # --- DATA: The Pure Eigenvalues (3, 1.5, 1) ---
        eigen_cols = [
            np.array([3.0, 0.0, 0.0]),
            np.array([0.0, 1.5, 0.0]),
            np.array([0.0, 0.0, 1.0])
        ]

        gold = "#FFD700"

        # 2. Create Vectors
        eigen_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=gold) for col in eigen_cols
        ])

        # Titles using Text (Safe from LaTeX errors)
        title = Text("Final Convergence: The Eigenvalues", color=gold, font_size=24)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        # 3. Animation
        self.play(Create(eigen_arrows), run_time=3)
        self.wait(1)

        # Label the lengths using standard Text
        # We use a simple 'L' to represent Lambda
        labels = VGroup(
            Text("L1 = 3.0", color=gold, font_size=20),
            Text("L2 = 1.5", color=gold, font_size=20),
            Text("L3 = 1.0", color=gold, font_size=20)
        )

        # Position labels at the bottom of the screen
        for i, label in enumerate(labels):
            self.add_fixed_in_frame_mobjects(label.to_edge(DOWN).shift(RIGHT * (i - 1) * 3))

        # 4. Final Victory Rotation
        self.begin_ambient_camera_rotation(rate=0.5)
        self.wait(4 * PI)
        self.stop_ambient_camera_rotation()

        self.wait(1)

Manim Community v0.20.1